# ROBERT Context Extractor

This notebook reads the output files from a completed ROBERT run and produces a single structured file called `run_context.json`.

That JSON file is the foundation for everything that comes next: diagnosing why a score is high or low, and producing plain-language explanations for chemists.

**What this notebook does:**
- Points to an archived ROBERT run folder (created by `robert_run_wrapper.ipynb`)
- Reads `PREDICT_data.dat` and `VERIFY_data.dat` (and optionally `CURATE_data.dat`)
- Extracts numbers and flags from those text files in a reliable, null-safe way
- Writes a `run_context.json` to the same run folder

**What this notebook does NOT do:**
- Re-run ROBERT
- Diagnose the score (that is the next notebook)
- Call any AI or external service

**Audience note:** The cells below include plain-language explanations before each code block. You do not need to read or edit the code to use this notebook — only Cell 3 needs to be updated for each new run.

## Cell 2 Guide: Point to a Run Folder

This is the only cell you need to edit for each new run.

Set `RUN_FOLDER` to the path of the archived run you want to inspect.
You will find archived run folders inside `agent/run_archive/` — each one is named
with a timestamp and the dataset name, for example:

```
agent/run_archive/20260513_160812__Hvapor/
```

You can either:
- paste the full path (e.g. `"/Users/.../agent/run_archive/20260513_160812__Hvapor"`), or
- leave `RUN_FOLDER = None` to auto-select the most recent run in `run_archive/`.

The extracted `run_context.json` will be written into that same run folder.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re

# -----------------------------------------------------------------------
# EDIT THIS: path to the run folder you want to extract context from.
# Set to None to auto-select the most recently created run.
# -----------------------------------------------------------------------
RUN_FOLDER = None

# -----------------------------------------------------------------------
# Auto-resolve project root (same logic as the wrapper notebook)
# -----------------------------------------------------------------------
def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "AGENTS.md").exists() and (candidate / "robert").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not infer project root. "
        "Expected a folder containing AGENTS.md and robert/."
    )

PROJECT_ROOT = resolve_project_root()
RUNS_ROOT = PROJECT_ROOT / "agent" / "run_archive"

if RUN_FOLDER is None:
    # Auto-select the most recently created run folder
    all_runs = sorted(
        [p for p in RUNS_ROOT.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not all_runs:
        raise FileNotFoundError(
            f"No run folders found in {RUNS_ROOT}. "
            "Run agent/robert_run_wrapper.ipynb first."
        )
    RUN_FOLDER = all_runs[0]
    print(f"Auto-selected most recent run: {RUN_FOLDER.name}")
else:
    RUN_FOLDER = Path(RUN_FOLDER).resolve()

if not RUN_FOLDER.exists():
    raise FileNotFoundError(f"Run folder not found: {RUN_FOLDER}")

OUTPUTS_DIR = RUN_FOLDER / "outputs"

print(f"Project root : {PROJECT_ROOT}")
print(f"Run folder   : {RUN_FOLDER}")
print(f"Outputs dir  : {OUTPUTS_DIR}")

Auto-selected most recent run: 20260513_160812__Hvapor
Project root : /Users/cjcscha/ROBERT/helper_rob/robert
Run folder   : /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260513_160812__Hvapor
Outputs dir  : /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260513_160812__Hvapor/outputs


## Cell 4 Guide: Check Which Output Files Are Present

ROBERT produces four output folders: `CURATE`, `GENERATE`, `VERIFY`, and `PREDICT`.
If a run stopped early, some folders may be missing.

This cell checks which `.dat` files exist and sets availability flags used throughout the rest of the notebook.
It also reads the `run_manifest.json` if it exists, which records the exact command used.

In [2]:
# Locate the key .dat files inside the outputs/ folder
PREDICT_DAT  = OUTPUTS_DIR / "PREDICT" / "PREDICT_data.dat"
VERIFY_DAT   = OUTPUTS_DIR / "VERIFY"  / "VERIFY_data.dat"
CURATE_DAT   = OUTPUTS_DIR / "CURATE"  / "CURATE_data.dat"
GENERATE_DAT = OUTPUTS_DIR / "GENERATE" / "GENERATE_data.dat"
MANIFEST_JSON = RUN_FOLDER / "run_manifest.json"

# Read each file into a list of lines, or None if the file is missing.
# This approach means downstream cells never crash on a missing file.
def read_dat(path: Path):
    """Return list of lines from a .dat file, or None if missing/unreadable."""
    if not path.exists():
        return None
    try:
        return path.read_text(encoding="utf-8").splitlines()
    except Exception as e:
        print(f"WARNING: could not read {path}: {e}")
        return None

predict_lines  = read_dat(PREDICT_DAT)
verify_lines   = read_dat(VERIFY_DAT)
curate_lines   = read_dat(CURATE_DAT)
generate_lines = read_dat(GENERATE_DAT)

# Set availability flags for the run_context.json
avail_predict  = predict_lines  is not None
avail_verify   = verify_lines   is not None
avail_curate   = curate_lines   is not None
avail_generate = generate_lines is not None

# Read run manifest for provenance
manifest_data = None
if MANIFEST_JSON.exists():
    try:
        manifest_data = json.loads(MANIFEST_JSON.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"WARNING: could not read manifest: {e}")

print(f"PREDICT_data.dat  : {'FOUND' if avail_predict  else 'MISSING'}")
print(f"VERIFY_data.dat   : {'FOUND' if avail_verify   else 'MISSING'}")
print(f"CURATE_data.dat   : {'FOUND' if avail_curate   else 'MISSING'}")
print(f"GENERATE_data.dat : {'FOUND' if avail_generate else 'MISSING'}")
print(f"run_manifest.json : {'FOUND' if manifest_data is not None else 'MISSING'}")

PREDICT_data.dat  : FOUND
VERIFY_data.dat   : FOUND
CURATE_data.dat   : FOUND
GENERATE_data.dat : FOUND
run_manifest.json : FOUND


## Cell 6 Guide: Parser Helper

This cell defines a small helper function used throughout the notebook.

It contains a shared utility (`safe_float`) and the function that splits a `.dat` file
into its two logical sections: one for the model trained on all variables (`No PFI`)
and one for the model trained on only the most important variables (`PFI` — Permutation
Feature Importance filter).

Most ROBERT runs produce both variants. The extractor handles each one independently
so they can be compared.

In [3]:
def safe_float(value_str, warnings_list=None, label=""):
    """
    Convert a string to float. Returns None if conversion fails.
    Appends a descriptive message to warnings_list if provided.
    """
    if value_str is None:
        return None
    try:
        return float(value_str.strip())
    except (ValueError, AttributeError):
        msg = f"Could not convert to float: {label!r} = {value_str!r}"
        if warnings_list is not None:
            warnings_list.append(msg)
        return None


def safe_int(value_str, warnings_list=None, label=""):
    """
    Convert a string to int. Returns None if conversion fails.
    """
    if value_str is None:
        return None
    try:
        return int(value_str.strip())
    except (ValueError, AttributeError):
        msg = f"Could not convert to int: {label!r} = {value_str!r}"
        if warnings_list is not None:
            warnings_list.append(msg)
        return None


def split_into_blocks(lines, no_pfi_marker, pfi_marker):
    """
    Split a list of .dat file lines into two blocks:
    - 'no_pfi': lines in the No PFI section
    - 'pfi': lines in the PFI section

    Returns a dict: {'no_pfi': [...], 'pfi': [...]}
    Each value is an empty list if the section was not found.
    """
    blocks = {"no_pfi": [], "pfi": []}
    if lines is None:
        return blocks

    current = None
    for line in lines:
        if no_pfi_marker in line:
            current = "no_pfi"
        elif pfi_marker in line:
            current = "pfi"
        if current is not None:
            blocks[current].append(line)

    return blocks


# The section headers used in PREDICT_data.dat and VERIFY_data.dat
NO_PFI_MARKER = "Starting model with all variables (No PFI)"
PFI_MARKER    = "Starting model with PFI filter"

print("Helper functions defined.")

Helper functions defined.


## Cell 8 Guide: Parse PREDICT_data.dat

This is the most information-rich file. For each model variant (No PFI and PFI),
ROBERT records:

- which descriptors (molecular features) were used,
- how many training and test compounds were used,
- cross-validation (CV) and test-set performance (R², MAE, RMSE for regression;
  MCC for classification),
- how uniform the data distribution was across the prediction range,
- which compounds were flagged as outliers.

The code extracts each of these values using pattern matching on the text lines.
If a line is missing or formatted differently (for example in a classification run),
the corresponding field is set to `None` rather than crashing.

In [4]:
def parse_predict_block(block_lines, warnings):
    """
    Extract performance metrics from one section (No PFI or PFI) of PREDICT_data.dat.

    Returns a dict with the fields defined in the V1 schema, plus bonus
    diagnostic fields (descriptors, n_train, n_test, outlier counts, etc.).
    All values are None if the corresponding line was not found.
    """
    result = {
        "cv_type"            : None,  # e.g. "10x 5-fold CV"
        "points_descp_ratio" : None,  # e.g. "106:3" (train:descriptors)
        "r2_cv"              : None,  # R² from cross-validation (or MCC for classification)
        "r2_test"            : None,  # R² on held-out test set (or MCC)
        "rmse_cv"            : None,  # RMSE from CV (null for classification)
        "rmse_test"          : None,  # RMSE on test set (null for classification)
        "mae_cv"             : None,
        "mae_test"           : None,
        "avg_sd_test"        : None,  # average prediction uncertainty (SD) on test
        "y_min"              : None,  # minimum y value in training set
        "y_max"              : None,  # maximum y value in training set
        "y_range"            : None,  # total span of y values
        "n_train"            : None,  # number of training+CV points
        "n_test"             : None,  # number of test points
        "n_descriptors"      : None,  # number of descriptors used
        "descriptors"        : None,  # list of descriptor names
        "model"              : None,  # model type (RF, GB, NN, MVL)
        "target"             : None,  # target property name
        "kfold"              : None,  # k value for cross-validation
        "cv_repeats"         : None,  # how many times CV was repeated
        "train_outlier_count": None,
        "train_outlier_pct"  : None,
        "test_outlier_count" : None,
        "test_outlier_pct"   : None,
        "quartile_counts"    : None,  # [Q1, Q2, Q3, Q4] point counts
    }

    if not block_lines:
        return result

    full_text = "\n".join(block_lines)

    # ----------------------------------------------------------------
    # Model and target metadata
    # ----------------------------------------------------------------
    m = re.search(r"-\s+Model:\s+(\S+)", full_text)
    if m:
        result["model"] = m.group(1).strip()

    m = re.search(r"-\s+Target value:\s+(.+)", full_text)
    if m:
        result["target"] = m.group(1).strip()

    m = re.search(r"-\s+k-fold CV:\s+(\d+)", full_text)
    if m:
        result["kfold"] = safe_int(m.group(1), warnings, "kfold")

    m = re.search(r"-\s+Repetitions CV:\s+(\d+)", full_text)
    if m:
        result["cv_repeats"] = safe_int(m.group(1), warnings, "cv_repeats")

    # ----------------------------------------------------------------
    # Descriptor list
    # ----------------------------------------------------------------
    m = re.search(r"-\s+Descriptors:\s+(\[.+?\])", full_text)
    if m:
        try:
            result["descriptors"] = json.loads(m.group(1).replace("'", '"'))
        except Exception:
            result["descriptors"] = m.group(1)  # keep raw string if parse fails

    # ----------------------------------------------------------------
    # Training and test point counts
    # ----------------------------------------------------------------
    m = re.search(r"-\s+Training points:\s+(\d+)", full_text)
    if m:
        result["n_train"] = safe_int(m.group(1), warnings, "n_train")

    m = re.search(r"-\s+Test points:\s+(\d+)", full_text)
    if m:
        result["n_test"] = safe_int(m.group(1), warnings, "n_test")

    # ----------------------------------------------------------------
    # Points-to-descriptors ratio (training data density)
    # ----------------------------------------------------------------
    m = re.search(r"Proportion \(train\+valid\.\) points:descriptors\s*=\s*([\d]+:[\d]+)", full_text)
    if m:
        result["points_descp_ratio"] = m.group(1)

    m = re.search(r"-\s+Number of descriptors\s*=\s*(\d+)", full_text)
    if m:
        result["n_descriptors"] = safe_int(m.group(1), warnings, "n_descriptors")

    # ----------------------------------------------------------------
    # Cross-validation performance
    # Format (regression): "10x 5-fold CV : R2 = 0.77, MAE = 4.7, RMSE = 5.7"
    # Format (classification): "10x 5-fold CV : MCC = 0.77, ..."
    # ----------------------------------------------------------------
    cv_pattern_reg = r"(\d+)x\s+\d+-fold CV\s*:\s+R2\s*=\s*([\d.eE+\-]+),\s*MAE\s*=\s*([\d.eE+\-]+),\s*RMSE\s*=\s*([\d.eE+\-]+)"
    cv_pattern_clas = r"(\d+)x\s+\d+-fold CV\s*:\s+MCC\s*=\s*([\d.eE+\-]+)"

    m = re.search(cv_pattern_reg, full_text)
    if m:
        repeats_check = m.group(1)
        kfold_check   = re.search(r"(\d+)x\s+(\d+)-fold CV", full_text)
        if kfold_check:
            result["cv_type"] = f"{kfold_check.group(1)}x {kfold_check.group(2)}-fold CV"
        result["r2_cv"]   = safe_float(m.group(2), warnings, "r2_cv")
        result["mae_cv"]  = safe_float(m.group(3), warnings, "mae_cv")
        result["rmse_cv"] = safe_float(m.group(4), warnings, "rmse_cv")
    else:
        m = re.search(cv_pattern_clas, full_text)
        if m:
            result["r2_cv"] = safe_float(m.group(2), warnings, "mcc_cv")  # MCC stored as r2_cv per schema
            kfold_check = re.search(r"(\d+)x\s+(\d+)-fold CV", full_text)
            if kfold_check:
                result["cv_type"] = f"{kfold_check.group(1)}x {kfold_check.group(2)}-fold CV"

    # ----------------------------------------------------------------
    # Test set performance
    # Format (regression): "-  Test : R2 = 0.78, MAE = 4.7, RMSE = 5.5"
    # ----------------------------------------------------------------
    test_pattern_reg  = r"-\s+Test\s*:\s+R2\s*=\s*([\d.eE+\-]+),\s*MAE\s*=\s*([\d.eE+\-]+),\s*RMSE\s*=\s*([\d.eE+\-]+)"
    test_pattern_clas = r"-\s+Test\s*:\s+MCC\s*=\s*([\d.eE+\-]+)"

    m = re.search(test_pattern_reg, full_text)
    if m:
        result["r2_test"]   = safe_float(m.group(1), warnings, "r2_test")
        result["mae_test"]  = safe_float(m.group(2), warnings, "mae_test")
        result["rmse_test"] = safe_float(m.group(3), warnings, "rmse_test")
    else:
        m = re.search(test_pattern_clas, full_text)
        if m:
            result["r2_test"] = safe_float(m.group(1), warnings, "mcc_test")

    # ----------------------------------------------------------------
    # Average SD in test set (prediction uncertainty)
    # ----------------------------------------------------------------
    m = re.search(r"Average SD in test set\s*=\s*([\d.eE+\-]+)", full_text)
    if m:
        result["avg_sd_test"] = safe_float(m.group(1), warnings, "avg_sd_test")

    # ----------------------------------------------------------------
    # y-value range of the training set
    # ----------------------------------------------------------------
    m = re.search(
        r"y range of dataset \(train\+valid\.\)\s*=\s*([\d.eE+\-]+) to ([\d.eE+\-]+),\s*total ([\d.eE+\-]+)",
        full_text
    )
    if m:
        result["y_min"]   = safe_float(m.group(1), warnings, "y_min")
        result["y_max"]   = safe_float(m.group(2), warnings, "y_max")
        result["y_range"] = safe_float(m.group(3), warnings, "y_range")

    # ----------------------------------------------------------------
    # Outliers in training and test sets
    # ----------------------------------------------------------------
    m = re.search(r"Train:\s+(\d+) outliers out of (\d+) datapoints \(([\d.]+)%\)", full_text)
    if m:
        result["train_outlier_count"] = safe_int(m.group(1), warnings, "train_outlier_count")
        result["train_outlier_pct"]   = safe_float(m.group(3), warnings, "train_outlier_pct")

    m = re.search(r"Test:\s+(\d+) outliers out of (\d+) datapoints \(([\d.]+)%\)", full_text)
    if m:
        result["test_outlier_count"] = safe_int(m.group(1), warnings, "test_outlier_count")
        result["test_outlier_pct"]   = safe_float(m.group(3), warnings, "test_outlier_pct")

    # ----------------------------------------------------------------
    # Quartile distribution of y values
    # ----------------------------------------------------------------
    m = re.search(r"Q1:\s*(\d+),\s*Q2:\s*(\d+),\s*Q3:\s*(\d+),\s*Q4:\s*(\d+)", full_text)
    if m:
        result["quartile_counts"] = [
            safe_int(m.group(1)), safe_int(m.group(2)),
            safe_int(m.group(3)), safe_int(m.group(4))
        ]

    return result


# Run the parser on both blocks
predict_warnings = []

if avail_predict:
    predict_blocks = split_into_blocks(predict_lines, NO_PFI_MARKER, PFI_MARKER)
    predict_no_pfi = parse_predict_block(predict_blocks["no_pfi"], predict_warnings)
    predict_pfi    = parse_predict_block(predict_blocks["pfi"],    predict_warnings)
    avail_pfi      = bool(predict_blocks["pfi"])  # True if PFI section exists
    avail_test_set = predict_no_pfi["r2_test"] is not None

    # Top-level metadata: model and pred_type come from the No PFI block
    ml_model  = predict_no_pfi["model"]
    # Detect prediction type from the CV line format
    # (R2 = regression, MCC = classification)
    if re.search(r"\d+x\s+\d+-fold CV\s*:\s+R2\s*=", "\n".join(predict_lines)):
        pred_type = "reg"
    elif re.search(r"\d+x\s+\d+-fold CV\s*:\s+MCC\s*=", "\n".join(predict_lines)):
        pred_type = "clas"
    else:
        pred_type = None
        predict_warnings.append("Could not detect pred_type from CV line format.")
else:
    predict_no_pfi = {k: None for k in [
        "cv_type", "points_descp_ratio", "r2_cv", "r2_test",
        "rmse_cv", "rmse_test", "mae_cv", "mae_test", "avg_sd_test",
        "y_min", "y_max", "y_range", "n_train", "n_test",
        "n_descriptors", "descriptors", "model", "target", "kfold",
        "cv_repeats", "train_outlier_count", "train_outlier_pct",
        "test_outlier_count", "test_outlier_pct", "quartile_counts"
    ]}
    predict_pfi    = dict(predict_no_pfi)  # same null shape
    avail_pfi      = False
    avail_test_set = False
    ml_model       = None
    pred_type      = None

print(f"Prediction type detected : {pred_type}")
print(f"ML model                 : {ml_model}")
print(f"PFI block present        : {avail_pfi}")
print(f"Test set metrics present : {avail_test_set}")
print()
print("--- No PFI model summary ---")
print(f"  CV  : R2={predict_no_pfi['r2_cv']}, RMSE={predict_no_pfi['rmse_cv']}, MAE={predict_no_pfi['mae_cv']}")
print(f"  Test: R2={predict_no_pfi['r2_test']}, RMSE={predict_no_pfi['rmse_test']}, MAE={predict_no_pfi['mae_test']}")
print(f"  Descriptors ({predict_no_pfi['n_descriptors']}): {predict_no_pfi['descriptors']}")
print(f"  Train N={predict_no_pfi['n_train']}, Test N={predict_no_pfi['n_test']}")
print(f"  Train outliers: {predict_no_pfi['train_outlier_count']} ({predict_no_pfi['train_outlier_pct']}%)",
      f"  Test outliers: {predict_no_pfi['test_outlier_count']} ({predict_no_pfi['test_outlier_pct']}%)")
print()
print("--- PFI model summary ---")
print(f"  CV  : R2={predict_pfi['r2_cv']}, RMSE={predict_pfi['rmse_cv']}, MAE={predict_pfi['mae_cv']}")
print(f"  Test: R2={predict_pfi['r2_test']}, RMSE={predict_pfi['rmse_test']}, MAE={predict_pfi['mae_test']}")
print(f"  Descriptors ({predict_pfi['n_descriptors']}): {predict_pfi['descriptors']}")
if predict_warnings:
    print(f"\nPREDICT parser warnings: {predict_warnings}")

Prediction type detected : reg
ML model                 : MVL
PFI block present        : True
Test set metrics present : True

--- No PFI model summary ---
  CV  : R2=0.77, RMSE=5.7, MAE=4.7
  Test: R2=0.78, RMSE=5.5, MAE=4.7
  Descriptors (3): ['Hardness', 'SGBP', 'g3']
  Train N=106, Test N=27
  Train outliers: 4 (3.8%)   Test outliers: 0 (0.0%)

--- PFI model summary ---
  CV  : R2=0.77, RMSE=5.7, MAE=4.7
  Test: R2=0.79, RMSE=5.2, MAE=4.5
  Descriptors (2): ['SGBP', 'Hardness']


## Cell 10 Guide: Parse VERIFY_data.dat

The VERIFY module tests whether the ML model is better than three simple baselines:

- `y_mean`: a model that predicts the mean value for everything (trivial baseline),
- `y_shuffle`: a model trained on randomly scrambled target values (random baseline),
- `onehot`: a model trained on arbitrary one-hot labels (structure-free baseline).

If your model cannot beat these baselines, it is essentially memorizing noise.

VERIFY also tests whether the model can predict the held-out extremes of the dataset
(sorted cross-validation), which reveals whether the model extrapolates or just interpolates.

This cell extracts: how many tests were passed, failed, or unclear — and the RMSE
values from sorted cross-validation.

In [5]:
def parse_verify_block(block_lines, warnings):
    """
    Extract verification test results from one section (No PFI or PFI) of VERIFY_data.dat.

    Returns a dict with:
    - failed_tests: count of FAILED baseline tests
    - unclear_tests: count of UNCLEAR baseline tests
    - passed_tests: count of PASSED baseline tests
    - test_results: list of raw result strings (e.g. 'y_mean: PASSED')
    - flawed_mod_score: penalty score (0 to -6; 0 = all passed, negative = problems)
    - sorted_cv_rmse: list of RMSE values from sorted 5-fold CV (or None)
    - sorted_cv_r2: list of R2 values from sorted 5-fold CV (or None)
    - cv_rmse_original: original CV RMSE used as the threshold baseline
    - threshold_15: 15% above original RMSE (low alert threshold)
    - threshold_30: 30% above original RMSE (high alert threshold)
    """
    result = {
        "failed_tests"      : None,
        "unclear_tests"     : None,
        "passed_tests"      : None,
        "test_results"      : None,
        "flawed_mod_score"  : None,
        "sorted_cv_rmse"    : None,
        "sorted_cv_r2"      : None,
        "cv_rmse_original"  : None,
        "threshold_15"      : None,
        "threshold_30"      : None,
    }

    if not block_lines:
        return result

    full_text = "\n".join(block_lines)

    # ----------------------------------------------------------------
    # Original RMSE and thresholds
    # Line format: "Original RMSE (10x 5-fold CV) 5.7 + 15% & 30% threshold = 6.5 & 7.4"
    # ----------------------------------------------------------------
    m = re.search(
        r"Original (?:RMSE|MCC) \(.*?\)\s+([\d.eE+\-]+)\s+\+\s+15%\s*&\s*30%\s*threshold\s*=\s*([\d.eE+\-]+)\s*&\s*([\d.eE+\-]+)",
        full_text
    )
    if m:
        result["cv_rmse_original"] = safe_float(m.group(1), warnings, "cv_rmse_original")
        result["threshold_15"]     = safe_float(m.group(2), warnings, "threshold_15")
        result["threshold_30"]     = safe_float(m.group(3), warnings, "threshold_30")

    # ----------------------------------------------------------------
    # Individual baseline test results
    # Lines look like:
    #   "o y_mean: PASSED, RMSE = 1.2e+01, higher than thresholds"
    #   "x y_shuffle: FAILED, RMSE = ..."
    # ----------------------------------------------------------------
    test_pattern = re.compile(
        r"[ox\-]\s+(y_mean|y_shuffle|onehot):\s+(PASSED|UNCLEAR|FAILED)"
    )
    matches = test_pattern.findall(full_text)

    if matches:
        result["test_results"] = [f"{name}: {status}" for name, status in matches]
        result["passed_tests"]  = sum(1 for _, s in matches if s == "PASSED")
        result["unclear_tests"] = sum(1 for _, s in matches if s == "UNCLEAR")
        result["failed_tests"]  = sum(1 for _, s in matches if s == "FAILED")
        # Flawed model score: each UNCLEAR costs 1 point, each FAILED costs 2 points
        result["flawed_mod_score"] = (
            -1 * result["unclear_tests"] +
            -2 * result["failed_tests"]
        )
    else:
        result["test_results"] = []
        result["passed_tests"]  = 0
        result["unclear_tests"] = 0
        result["failed_tests"]  = 0
        result["flawed_mod_score"] = 0

    # ----------------------------------------------------------------
    # Sorted cross-validation results
    # Line format:
    #   "- Sorted 5-fold CV : R2 = [0.64, 0.02, ...], MAE = [...], RMSE = [...]"
    # ----------------------------------------------------------------
    m_sorted = re.search(r"-\s+Sorted \d+-fold CV\s*:.*?RMSE\s*=\s*(\[[^\]]+\])", full_text)
    if m_sorted:
        try:
            result["sorted_cv_rmse"] = json.loads(m_sorted.group(1))
        except Exception:
            warnings.append(f"Could not parse sorted CV RMSE list: {m_sorted.group(1)}")

    m_r2 = re.search(r"-\s+Sorted \d+-fold CV\s*:.*?R2\s*=\s*(\[[^\]]+\])", full_text)
    if m_r2:
        try:
            result["sorted_cv_r2"] = json.loads(m_r2.group(1))
        except Exception:
            warnings.append(f"Could not parse sorted CV R2 list: {m_r2.group(1)}")

    return result


# Run the parser on both VERIFY blocks
verify_warnings = []

if avail_verify:
    verify_blocks  = split_into_blocks(verify_lines, NO_PFI_MARKER, PFI_MARKER)
    verify_no_pfi  = parse_verify_block(verify_blocks["no_pfi"], verify_warnings)
    verify_pfi     = parse_verify_block(verify_blocks["pfi"],    verify_warnings)
else:
    empty_verify = {
        "failed_tests": None, "unclear_tests": None, "passed_tests": None,
        "test_results": None, "flawed_mod_score": None,
        "sorted_cv_rmse": None, "sorted_cv_r2": None,
        "cv_rmse_original": None, "threshold_15": None, "threshold_30": None,
    }
    verify_no_pfi = dict(empty_verify)
    verify_pfi    = dict(empty_verify)

print("--- No PFI verification ---")
print(f"  Baseline tests: {verify_no_pfi['test_results']}")
print(f"  Passed={verify_no_pfi['passed_tests']}, Unclear={verify_no_pfi['unclear_tests']}, Failed={verify_no_pfi['failed_tests']}")
print(f"  Flawed model score contribution: {verify_no_pfi['flawed_mod_score']}")
print(f"  Sorted CV RMSE: {verify_no_pfi['sorted_cv_rmse']}")
print()
print("--- PFI verification ---")
print(f"  Baseline tests: {verify_pfi['test_results']}")
print(f"  Passed={verify_pfi['passed_tests']}, Unclear={verify_pfi['unclear_tests']}, Failed={verify_pfi['failed_tests']}")
print(f"  Flawed model score contribution: {verify_pfi['flawed_mod_score']}")
if verify_warnings:
    print(f"\nVERIFY parser warnings: {verify_warnings}")

--- No PFI verification ---
  Baseline tests: ['y_mean: PASSED', 'y_shuffle: PASSED', 'onehot: PASSED']
  Passed=3, Unclear=0, Failed=0
  Flawed model score contribution: 0
  Sorted CV RMSE: [7.57, 5.32, 6.46, 4.99, 8.76]

--- PFI verification ---
  Baseline tests: ['y_mean: PASSED', 'y_shuffle: PASSED', 'onehot: PASSED']
  Passed=3, Unclear=0, Failed=0
  Flawed model score contribution: 0


## Cell 12 Guide: Parse CURATE_data.dat and run_manifest.json

The CURATE module cleans and filters the input dataset before model training. It:
- removes duplicate compounds,
- removes descriptors (molecular features) that are nearly perfectly correlated with each other,
- applies recursive feature elimination (RFECV) if the dataset is large enough.

This cell extracts the initial dataset size, how many features were available before curation,
and how many were kept after filtering.

It also reads the `run_manifest.json` to capture the exact ROBERT command used.

In [6]:
def parse_curate_dat(lines, warnings):
    """
    Extract dataset and feature counts from CURATE_data.dat.

    Returns a dict with initial and final feature/datapoint counts,
    plus any curation warning lines.
    """
    result = {
        "n_initial"          : None,  # datapoints at start
        "n_final"            : None,  # datapoints after curation (duplicates removed etc.)
        "n_dropped"          : None,  # derived: n_initial - n_final
        "n_features_initial" : None,  # accepted descriptors before filtering
        "n_features_final"   : None,  # descriptors remaining after filtering
        "n_features_removed" : None,  # descriptors removed by correlation/RFECV
        "ignored_descriptors": None,  # columns set aside (name columns, etc.)
        "curate_warnings"    : [],
    }

    if not lines:
        return result

    full_text = "\n".join(lines)

    # ----------------------------------------------------------------
    # Initial database load (first occurrence of the database summary)
    # These lines describe the input file BEFORE any filtering.
    # ----------------------------------------------------------------
    m = re.search(r"Database .+? loaded successfully", full_text)
    if m:
        # Everything from the first database summary
        first_load_text = full_text[m.start():m.start()+400]

        m2 = re.search(r"(\d+) datapoints", first_load_text)
        if m2:
            result["n_initial"] = safe_int(m2.group(1), warnings, "n_initial")

        m2 = re.search(r"(\d+) accepted descriptors", first_load_text)
        if m2:
            result["n_features_initial"] = safe_int(m2.group(1), warnings, "n_features_initial")

        m2 = re.search(r"(\d+) ignored descriptors", first_load_text)
        if m2:
            result["ignored_descriptors"] = safe_int(m2.group(1), warnings, "ignored_descriptors")

    # ----------------------------------------------------------------
    # Total descriptors removed by the correlation filter
    # ----------------------------------------------------------------
    m = re.search(r"Total:\s+(\d+) descriptors removed due to high correlation", full_text)
    if m:
        result["n_features_removed"] = safe_int(m.group(1), warnings, "n_features_removed")

    # ----------------------------------------------------------------
    # Final feature count (n_features_initial - n_features_removed)
    # ----------------------------------------------------------------
    if result["n_features_initial"] is not None:
        removed = result["n_features_removed"] or 0
        result["n_features_final"] = result["n_features_initial"] - removed

    # ----------------------------------------------------------------
    # Datapoints removed during curation
    # ----------------------------------------------------------------
    if re.search(r"No datapoints were removed", full_text):
        result["n_final"]   = result["n_initial"]
        result["n_dropped"] = 0
    else:
        # Count 'Excluded datapoints' lines if any
        dropped_matches = re.findall(r"(\d+) datapoints? (?:excluded|removed|discarded)", full_text)
        if dropped_matches:
            total_dropped = sum(safe_int(x) or 0 for x in dropped_matches)
            if result["n_initial"] is not None:
                result["n_final"]   = result["n_initial"] - total_dropped
                result["n_dropped"] = total_dropped

    # ----------------------------------------------------------------
    # Collect any warning-flagged lines (lines starting with 'x ')
    # ----------------------------------------------------------------
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("x "):
            result["curate_warnings"].append(stripped)

    return result


curate_warnings = []

if avail_curate:
    curate_info = parse_curate_dat(curate_lines, curate_warnings)
else:
    curate_info = {
        "n_initial": None, "n_final": None, "n_dropped": None,
        "n_features_initial": None, "n_features_final": None,
        "n_features_removed": None, "ignored_descriptors": None,
        "curate_warnings": [],
    }

# Manifest provenance
robert_command_used = None
dataset_csv_used    = None
if manifest_data:
    robert_command_used = manifest_data.get("robert_command")
    dataset_csv_used    = manifest_data.get("dataset_csv")

print("--- CURATE summary ---")
print(f"  Initial datapoints     : {curate_info['n_initial']}")
print(f"  Final datapoints       : {curate_info['n_final']}  (dropped: {curate_info['n_dropped']})")
print(f"  Initial features       : {curate_info['n_features_initial']}")
print(f"  Features after curation: {curate_info['n_features_final']}  (removed: {curate_info['n_features_removed']})")
print(f"  Ignored columns        : {curate_info['ignored_descriptors']}")
if curate_info["curate_warnings"]:
    print(f"  CURATE warnings        : {curate_info['curate_warnings']}")
print()
print("--- Provenance ---")
print(f"  Dataset CSV : {dataset_csv_used}")
print(f"  Command     : {' '.join(robert_command_used) if robert_command_used else 'not available'}")

--- CURATE summary ---
  Initial datapoints     : 133
  Final datapoints       : 133  (dropped: 0)
  Initial features       : 5
  Features after curation: 3  (removed: 2)
  Ignored columns        : 1
  CURATE warnings        : ['x The RFECV filter was not applied, there are less descriptors than one-third of the data points (3 <= 44)']

--- Provenance ---
  Dataset CSV : /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv
  Command     : python -m robert --names Name --y Hvapor --csv_name /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv


## Cell 14 Guide: Assemble and Write run_context.json

This cell combines everything extracted above into a single structured file: `run_context.json`.

The file is written to the run folder alongside the archived ROBERT output folders.
It follows the V1 schema defined in `agent/run_context_schema.md`.

The JSON file is then printed in a human-readable form so you can visually check
that the extraction looks correct.

If any field shows `null`, that means either the relevant ROBERT output was missing,
or the parser could not find that value in the output file.

In [7]:
# Collect all parser warnings together
all_warnings = predict_warnings + verify_warnings + curate_warnings

# Assemble the run_context dict following the V1 schema
run_context = {
    "schema_version" : "1.0",
    "extracted_at"   : datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "results_dir"    : str(RUN_FOLDER),
    "pred_type"      : pred_type,    # "reg", "clas", or null
    "ml_model"       : ml_model,     # e.g. "MVL", "RF"
    "dataset_csv"    : dataset_csv_used,
    "robert_command" : robert_command_used,

    # Which output files were found in this run folder
    "available": {
        "predict"  : avail_predict,
        "verify"   : avail_verify,
        "curate"   : avail_curate,
        "generate" : avail_generate,
        "test_set" : avail_test_set,
        "pfi"      : avail_pfi,
    },

    # PREDICT metrics — one sub-dict per model variant
    "predict": {
        "no_pfi": {
            "cv_type"            : predict_no_pfi["cv_type"],
            "points_descp_ratio" : predict_no_pfi["points_descp_ratio"],
            "r2_cv"              : predict_no_pfi["r2_cv"],
            "r2_test"            : predict_no_pfi["r2_test"],
            "rmse_cv"            : predict_no_pfi["rmse_cv"],
            "rmse_test"          : predict_no_pfi["rmse_test"],
            "mae_cv"             : predict_no_pfi["mae_cv"],
            "mae_test"           : predict_no_pfi["mae_test"],
            "avg_sd_test"        : predict_no_pfi["avg_sd_test"],
            "n_train"            : predict_no_pfi["n_train"],
            "n_test"             : predict_no_pfi["n_test"],
            "n_descriptors"      : predict_no_pfi["n_descriptors"],
            "descriptors"        : predict_no_pfi["descriptors"],
            "model"              : predict_no_pfi["model"],
            "target"             : predict_no_pfi["target"],
            "kfold"              : predict_no_pfi["kfold"],
            "cv_repeats"         : predict_no_pfi["cv_repeats"],
            "y_min"              : predict_no_pfi["y_min"],
            "y_max"              : predict_no_pfi["y_max"],
            "y_range"            : predict_no_pfi["y_range"],
            "train_outlier_count": predict_no_pfi["train_outlier_count"],
            "train_outlier_pct"  : predict_no_pfi["train_outlier_pct"],
            "test_outlier_count" : predict_no_pfi["test_outlier_count"],
            "test_outlier_pct"   : predict_no_pfi["test_outlier_pct"],
            "quartile_counts"    : predict_no_pfi["quartile_counts"],
        },
        "pfi": {
            "cv_type"            : predict_pfi["cv_type"],
            "points_descp_ratio" : predict_pfi["points_descp_ratio"],
            "r2_cv"              : predict_pfi["r2_cv"],
            "r2_test"            : predict_pfi["r2_test"],
            "rmse_cv"            : predict_pfi["rmse_cv"],
            "rmse_test"          : predict_pfi["rmse_test"],
            "mae_cv"             : predict_pfi["mae_cv"],
            "mae_test"           : predict_pfi["mae_test"],
            "avg_sd_test"        : predict_pfi["avg_sd_test"],
            "n_train"            : predict_pfi["n_train"],
            "n_test"             : predict_pfi["n_test"],
            "n_descriptors"      : predict_pfi["n_descriptors"],
            "descriptors"        : predict_pfi["descriptors"],
            "model"              : predict_pfi["model"],
            "target"             : predict_pfi["target"],
            "kfold"              : predict_pfi["kfold"],
            "cv_repeats"         : predict_pfi["cv_repeats"],
            "y_min"              : predict_pfi["y_min"],
            "y_max"              : predict_pfi["y_max"],
            "y_range"            : predict_pfi["y_range"],
            "train_outlier_count": predict_pfi["train_outlier_count"],
            "train_outlier_pct"  : predict_pfi["train_outlier_pct"],
            "test_outlier_count" : predict_pfi["test_outlier_count"],
            "test_outlier_pct"   : predict_pfi["test_outlier_pct"],
            "quartile_counts"    : predict_pfi["quartile_counts"],
        },
    },

    # VERIFY results — baseline tests and sorted CV
    "verify": {
        "no_pfi": {
            "failed_tests"     : verify_no_pfi["failed_tests"],
            "unclear_tests"    : verify_no_pfi["unclear_tests"],
            "passed_tests"     : verify_no_pfi["passed_tests"],
            "test_results"     : verify_no_pfi["test_results"],
            "flawed_mod_score" : verify_no_pfi["flawed_mod_score"],
            "sorted_cv_rmse"   : verify_no_pfi["sorted_cv_rmse"],
            "sorted_cv_r2"     : verify_no_pfi["sorted_cv_r2"],
            "cv_rmse_original" : verify_no_pfi["cv_rmse_original"],
            "threshold_15"     : verify_no_pfi["threshold_15"],
            "threshold_30"     : verify_no_pfi["threshold_30"],
        },
        "pfi": {
            "failed_tests"     : verify_pfi["failed_tests"],
            "unclear_tests"    : verify_pfi["unclear_tests"],
            "passed_tests"     : verify_pfi["passed_tests"],
            "test_results"     : verify_pfi["test_results"],
            "flawed_mod_score" : verify_pfi["flawed_mod_score"],
            "sorted_cv_rmse"   : verify_pfi["sorted_cv_rmse"],
            "sorted_cv_r2"     : verify_pfi["sorted_cv_r2"],
            "cv_rmse_original" : verify_pfi["cv_rmse_original"],
            "threshold_15"     : verify_pfi["threshold_15"],
            "threshold_30"     : verify_pfi["threshold_30"],
        },
    },

    # CURATE summary
    "curate": {
        "n_initial"          : curate_info["n_initial"],
        "n_final"            : curate_info["n_final"],
        "n_dropped"          : curate_info["n_dropped"],
        "n_features_initial" : curate_info["n_features_initial"],
        "n_features_final"   : curate_info["n_features_final"],
        "n_features_removed" : curate_info["n_features_removed"],
        "ignored_descriptors": curate_info["ignored_descriptors"],
        "warnings"           : curate_info["curate_warnings"],
    },

    # Score fields — populated by the next notebook (diagnose_score)
    "score": {
        "no_pfi" : None,  # null in V1: score reconstruction deferred to diagnose_score
        "pfi"    : None,
    },

    # Any issues encountered during extraction
    "parser_warnings": all_warnings,
}

# Write to file
output_path = RUN_FOLDER / "run_context.json"
output_path.write_text(json.dumps(run_context, indent=2), encoding="utf-8")

print(f"run_context.json written to:\n  {output_path}")
print()
print("Contents:")
print(json.dumps(run_context, indent=2))

run_context.json written to:
  /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260513_160812__Hvapor/run_context.json

Contents:
{
  "schema_version": "1.0",
  "extracted_at": "2026-05-13T21:19:28Z",
  "results_dir": "/Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260513_160812__Hvapor",
  "pred_type": "reg",
  "ml_model": "MVL",
  "dataset_csv": "/Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv",
  "robert_command": [
    "python",
    "-m",
    "robert",
    "--names",
    "Name",
    "--y",
    "Hvapor",
    "--csv_name",
    "/Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv"
  ],
  "available": {
    "predict": true,
    "verify": true,
    "curate": true,
    "generate": true,
    "test_set": true,
    "pfi": true
  },
  "predict": {
    "no_pfi": {
      "cv_type": "10x 5-fold CV",
      "points_descp_ratio": "106:3",
      "r2_cv": 0.77,
      "r2_test": 0.78,
      "rmse_cv": 5.7,
      "rmse_test": 5.5,
   

## Usage Notes

### How to use this notebook on a new run

1. Run `agent/robert_run_wrapper.ipynb` to generate and archive a ROBERT run.
2. Open this notebook and update `RUN_FOLDER` in Cell 3 to point at the new archive,
   or leave it as `None` to auto-select the latest run.
3. Run all cells from top to bottom.
4. Check the printed summary from each section for unexpected `None` values.
5. The `run_context.json` file is written into the run folder for downstream use.

### Interpreting null values

- `null` for any metric field means that line was not found in the `.dat` file.
- `null` for a whole block (e.g. `predict.pfi`) means the PFI section was not
  present in the run (possible if ROBERT stopped before PREDICT, or PFI was disabled).
- Check `parser_warnings` at the bottom of the JSON for any conversion issues.

### What comes next

The `run_context.json` produced here is the input for `agent/diagnose_score.ipynb`,
which will apply rule-based diagnostic flags to explain the ROBERT score.